# SleepFM — embeddings PhysioNet 2018 (GPU)

Fichier : `notebooks/sleepfm_embeddings_physionet2018.ipynb`. Sur Kaggle,
renommer le notebook importé avec ce même titre pour le retrouver.


Stadification (5 classes : Wake, N1, N2, N3, REM) avec le checkpoint officiel
SleepFM, sur de vrais enregistrements PhysioNet 2018.

**Sur Colab** : `Exécution > Modifier le type d'exécution > GPU T4`, puis
`Exécution > Tout exécuter`.

**Sur Kaggle** (même notebook, détecté automatiquement) — utile quand le quota
GPU de Colab est épuisé (~30 h de GPU par semaine sur Kaggle) :
1. `Settings > Accelerator > GPU T4 x2` (pas P100 : trop ancien pour PyTorch
   récent) et `Settings > Internet > On` (compte vérifié par téléphone).
2. Pour un long run (200 sujets, ~2 h 30) : `Save Version > Save & Run All
   (Commit)`. Le calcul tourne alors en arrière-plan, même navigateur fermé
   (jusqu'à 12 h), et les sorties restent dans l'onglet `Output` de la
   version. `Run All` en session interactive s'arrête si la session expire.
3. Optionnel, pour ne pas recalculer : publier un dossier d'embeddings déjà
   calculé comme Dataset Kaggle **privé**, puis `Add Input`. Il n'est copié
   que si son `manifest.json` a exactement le même contrat : checkpoint,
   prétraitement, canaux, fréquence. Les anciens embeddings (version 1, avant
   le 5 nov. 2026) ou ceux d'un autre checkpoint sont ignorés.
4. Les résultats, les prédictions et un zip des embeddings sont copiés dans
   `/kaggle/working`, à télécharger depuis `Output`.

Les sujets sont traités **par lots** : téléchargement → extraction → embeddings,
puis le brut est supprimé et seuls les embeddings (~5 Mo/sujet) sont gardés.
Avec Google Drive (Colab), une déconnexion ne fait rien perdre : relancer
reprend là où on s'était arrêté. Sur Kaggle, le cache ne survit pas à la
session : télécharger les sorties avant de la fermer.

PhysioNet 2018 est en accès libre (pas de DUA), donc pas de restriction à
l'utiliser sur Colab ou Kaggle — contrairement à MESA/SHHS/CHAT (NSRR).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : sur Colab, Exécution > Modifier le type d'exécution > GPU T4 ; sur Kaggle, Settings > Accelerator"
!df -h . | tail -1


## 1. Code et dépendances

In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
%cd {WORK}
REVISION = "origin/main"  # pour la confirmation : le commit exact, noté dans les sorties
![ -d sleepfm-pipeline ] || git clone -q https://github.com/sumarc183-design/sleepfm-pipeline.git
# Un découpage étendu par un run précédent est gardé hors du dépôt avant la mise à jour.
!cd sleepfm-pipeline && (git diff --quiet -- configs/split_physionet2018.json || cp configs/split_physionet2018.json "../split_avant_mise_a_jour_$(date +%s).json")
# Force la version voulue du code : un simple git pull peut échouer sans
# message (fichier modifié par un run précédent) et laisser l'ancien code.
!cd sleepfm-pipeline && git fetch -q && git reset -q --hard {REVISION} && git log --oneline -1
%cd {WORK}/sleepfm-pipeline
!pip install -q -r requirements.txt


## 2. Checkpoint officiel SleepFM (`best.pt`, ~84 Mo, versionné dans le dépôt officiel)

In [ ]:
![ -d third_party/sleepfm ] || git clone -q --depth 1 https://github.com/rthapa84/sleepfm-codebase.git third_party/sleepfm
!ls -lh third_party/sleepfm/sleepfm/checkpoint/best.pt

## 3. Paramètres

- `N_SUBJECTS` : les N premiers sujets d'entraînement PhysioNet 2018 (994 au
  total, pris par identifiant). Le découpage est figé
  (`configs/split_physionet2018.json`) : les sujets déjà répartis gardent
  leur rôle, les nouveaux sont répartis 80 / 20. Par défaut 500, soit 100
  nuits de test comme l'article. Avec les embeddings des 200 premiers en
  Dataset Kaggle (`Add Input`), seuls les 300 nouveaux sont calculés
  (~3 h 45 de GPU).
- `EMB_DIR` : dossier `physionet2018_v2`, pour ne pas mélanger ces embeddings
  (prétraitement version 2) avec les anciens (`physionet2018`, version 1).
- `EMBED_ONLY` (par défaut `True`) : le GPU ne calcule que les embeddings.
  Les sondes, qui n'ont pas besoin de GPU, tournent ensuite sur le Mac à
  partir du zip des embeddings : ~20-25 min de GPU économisées par run.
  `False` pour entraîner aussi les sondes ici.
- `USE_DRIVE` (Colab) : garde les embeddings sur Google Drive (~5 Mo/sujet)
  pour reprendre après une déconnexion. Colab demande l'autorisation d'accès
  au Drive. Sur Kaggle, pas de Drive.
- `CONFIRMATION` (par défaut `False`) : calcule seulement les embeddings des
  494 sujets réservés (`configs/reserved_physionet2018.json`), sans sonde, dans
  un dossier à part. **Ne le passer à `True` qu'après validation de
  `docs/PROTOCOLE_CONFIRMATION.md`**, et aucun score n'est calculé ici.


In [ ]:
import glob
import json
import shutil

N_SUBJECTS = 500  # 100 nuits de test (20 %), comme l'article SleepFM 2024
CONFIRMATION = False  # True seulement après validation du protocole de confirmation
SUBJECT_IDS = None  # sinon, les N_SUBJECTS premiers sujets
CACHE_NAME = "physionet2018_v2"
# True : le GPU ne sert qu'aux embeddings ; les sondes tournent ensuite sur le
# Mac (CPU) à partir du zip des embeddings. Économise ~20-25 min de GPU.
# False : sondes entraînées ici aussi (résultats complets dans ce notebook).
EMBED_ONLY = True
if CONFIRMATION:
    EMBED_ONLY = True  # aucune sonde, aucun score : voir docs/PROTOCOLE_CONFIRMATION.md
    SUBJECT_IDS = json.load(open("configs/reserved_physionet2018.json"))["subjects"]
    N_SUBJECTS = len(SUBJECT_IDS)
    CACHE_NAME = "physionet2018_reserved_v2"
EMBED_FLAG = "--embed-only" if EMBED_ONLY else ""
USE_DRIVE = not ON_KAGGLE   # Kaggle : pas de Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    EMB_DIR = f"/content/drive/MyDrive/sleepfm_embeddings/{CACHE_NAME}"
else:
    EMB_DIR = f"{WORK}/embeddings/{CACHE_NAME}"
os.makedirs(EMB_DIR, exist_ok=True)

if ON_KAGGLE and not CONFIRMATION:
    # /kaggle/input est en lecture seule : on copie dans le dossier de travail
    # les embeddings importés (Dataset Kaggle). Seules les sources au contrat
    # identique (checkpoint, prétraitement, canaux, fréquence) sont copiées ;
    # un fichier présent avec un autre contenu arrête le notebook.
    # Un Dataset créé depuis l'onglet Output peut garder le zip tel quel :
    # on le décompresse d'abord dans un dossier temporaire.
    import sys
    import zipfile
    from pathlib import Path
    sys.path.insert(0, os.getcwd())
    from run_baseline import embeddings_manifest
    from src.data.cache_contract import import_embeddings
    for k, archive in enumerate(glob.glob("/kaggle/input/**/embeddings_physionet2018_v2*.zip", recursive=True)):
        target = f"/kaggle/temp/imported_{k}"
        print(f"Décompression de {archive}")
        zipfile.ZipFile(archive).extractall(target)
    sources = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
    sources += glob.glob("/kaggle/temp/imported_*/**/manifest.json", recursive=True)
    expected = embeddings_manifest(Path("third_party/sleepfm/sleepfm/checkpoint/best.pt"))
    report = import_embeddings([Path(m) for m in sources], Path(EMB_DIR), expected)
    print(f"{report['copied']} fichier(s) d'embeddings importés depuis /kaggle/input")
    if sources and not report["copied"] and not glob.glob(f"{EMB_DIR}/*.npz"):
        print("ATTENTION : aucune source compatible, tous les sujets seront recalculés (voir les lignes « Ignoré »)")

print("Embeddings :", EMB_DIR)

## 4. Baseline : téléchargement + extraction par lots → embeddings (GPU) → sondes

Split **80/20 par sujet** (jamais par époque). Sondes sur les mêmes embeddings :
régression logistique pondérée (baseline, protocole officiel) et sa version
non pondérée, XGBoost (GPU), MLP, et un BiGRU qui lit la nuit entière
(contexte temporel). Pour chacune : accuracy, kappa, F1 macro, ROC-AUC, AUPRC, et
le **gap** train − test (un gap élevé = surapprentissage), plus les intervalles de
confiance à 95 % (bootstrap sur les sujets), la matrice de confusion et la
convergence de la régression logistique.

Un seul de MLP / XGBoost est gardé, choisi **sur le train uniquement** par
validation croisée 5 plis groupée par sujet (éliminé si gap F1 > 0,15 ; sinon
XGBoost seulement s'il gagne de plus de 0,02 en moyenne et sur 4 plis sur 5).
Le test n'évalue que les 4 sondes retenues, et ne sert à aucun choix.
Embeddings : ~1 h 15 par tranche de 100 sujets sur GPU. Comptez ensuite ~15 à 25 min pour les sondes sur GPU (dont la validation croisée),
~1 h à 1 h 30 sur CPU ; `--probes logistic` pour la baseline seule.

Si les embeddings sont déjà là (Drive ou Dataset Kaggle), seule cette étape
de sondes est refaite. Un fichier d'embeddings illisible (écriture
interrompue) est détecté, supprimé et recalculé automatiquement.


In [ ]:
import subprocess
import time

run_started = time.time()
log_path = f"results_physionet2018_{N_SUBJECTS}.txt"
command = [
    "python", "-u", "run_baseline.py",
    *(["--subjects-file", "configs/reserved_physionet2018.json"] if CONFIRMATION
      else ["--physionet-subjects", str(N_SUBJECTS)]),
    "--embeddings-dir", EMB_DIR,
    "--checkpoint", "third_party/sleepfm/sleepfm/checkpoint/best.pt",
    "--split-file", "configs/split_physionet2018.json",
    "--num-threads", "2",
    "--chunk-size", "8",
    "--batch-size", "128",
    "--results-json", f"results_physionet2018_{N_SUBJECTS}.json",
]
if EMBED_FLAG:
    command.append(EMBED_FLAG)
# Affiche et enregistre la sortie, en gardant le vrai code de retour du script.
with open(log_path, "w") as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"run_baseline.py a échoué (code {return_code}) : voir la sortie ci-dessus")


## 5. Garder le résultat

Avec `EMBED_ONLY = True`, télécharger surtout `embeddings_physionet2018_v2.zip`
(Kaggle) ou le dossier du Drive (Colab), puis, sur le Mac :

```bash
python run_baseline.py --physionet-subjects 200 --embeddings-dir embeddings/physionet2018_v2 \
    --checkpoint third_party/sleepfm/sleepfm/checkpoint/best.pt \
    --split-file configs/split_physionet2018.json --device cpu --results-json results/run.json
```

Sinon, télécharger `results_physionet2018_<N>.txt` (sortie lisible), `.json`
(métriques, sujets, réglages, provenance des embeddings), `.predictions.npz`
(prédictions du test époque par époque) et `embeddings_physionet2018_v2.zip`
(à garder pour relancer les sondes sans GPU) : sur Colab, la cellule
ci-dessous (les embeddings restent sur le Drive) ; sur Kaggle, le panneau
`Output` (fichiers copiés dans `/kaggle/working`).

**Lecture du chiffre** : la régression logistique pondérée sur embeddings
figés est le protocole officiel. L'article SleepFM 2024 évalue aussi
PhysioNet 2018 (section 4.6, tableau 8 : ROC-AUC 0,924, AUPRC 0,759, F1
0,700, 100 nuits de test) ; la sortie compare la baseline à ce tableau, stade
par stade. Le protocole diffère (nombre de nuits de test, découpage) : c'est
un ordre de grandeur, pas une reproduction stricte.

**Confirmation** (`CONFIRMATION = True`, après validation du protocole) :
télécharger `embeddings_physionet2018_reserved_v2.zip`, le dézipper dans
`embeddings/physionet2018_reserved_v2/`, puis lancer sur le Mac le seul passage
prévu, sans entraînement :
`python scripts/confirm_frozen_models.py --embeddings-dir embeddings/physionet2018_reserved_v2 --out results/confirmation_physionet2018`


In [ ]:
base = f"{WORK}/sleepfm-pipeline/results_physionet2018_{N_SUBJECTS}"
# Le run de CE notebook doit avoir produit son bilan : chaque sujet demandé a un
# seul résultat. Un cache partiel ou un ancien run ne suffit pas à annoncer un succès.
status_path = f"{EMB_DIR}/run_status.json"
if not os.path.exists(status_path) or os.path.getmtime(status_path) < run_started:
    raise RuntimeError("Le run n'a pas écrit son bilan : voir la sortie de la cellule précédente.")
status = json.load(open(status_path))
print(f"{len(status['embedded'])} sujet(s) d'embeddings prêts, {len(status['rejected'])} rejeté(s), "
      f"{len(status['requested'])} demandés, {len(status['not_requested'])} autres dans le dossier")
if not status["complete"] or len(status["requested"]) != N_SUBJECTS or not os.path.exists(f"{EMB_DIR}/manifest.json"):
    raise RuntimeError(f"Run incomplet : manquants {status['missing'][:5]}, doubles {status['both_embedded_and_rejected'][:5]}")
!python -m pip freeze > pip_freeze.txt
if EMBED_ONLY:
    outputs = [f"{base}.txt"]
else:
    outputs = [f"{base}.txt", f"{base}.json", f"{base}.predictions.npz"]
    if not os.path.exists(f"{base}.json") or os.path.getmtime(f"{base}.json") < run_started:
        raise RuntimeError(f"Le run n'a pas écrit {base}.json : voir la sortie de la cellule 4.")
outputs += ["configs/split_physionet2018.json", "pip_freeze.txt", status_path]  # pour vérification
if ON_KAGGLE:
    for f in outputs:
        shutil.copy(f, WORK)  # visibles à la racine du panneau Output
    shutil.make_archive(f"{WORK}/embeddings_{CACHE_NAME}", "zip", EMB_DIR)
    shutil.rmtree(f"{WORK}/embeddings")  # le zip suffit : évite de doubler la taille des sorties
    print("Résultats, prédictions et embeddings (zip) dans /kaggle/working : panneau Output, à droite")
else:
    from google.colab import files
    for f in outputs:
        files.download(f)